In [115]:
import sqlite3
import pandas as pd

In [116]:
conn = sqlite3.connect("download.db")
Qu1_sql = """SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) as total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id;
"""
reslt_Qu1 = pd.read_sql_query(Qu1_sql, conn)
print("\nQ1 Result:\n", reslt_Qu1)



Q1 Result:
     member_id first_name last_name  total_checkouts
0        1001      Salma   Ibrahim                1
1        1002      Fares     Saleh                2
2        1003     Bassel    Hegazy                9
3        1004      Fares     Wahba                0
4        1005    Youssef     Halim                3
..        ...        ...       ...              ...
75       1076       Dina     Wahba                7
76       1077       Lina    Rashad                6
77       1078     Habiba     Osman                0
78       1079       Rana     Osman               10
79       1080     Bassel     Wahba                2

[80 rows x 4 columns]


In [117]:
Qu2_sql= "SELECT * FROM books WHERE author LIKE '%Aya%';"
reslt_Qu2 = pd.read_sql_query(Qu2_sql, conn)
print("\nQ2 Result:\n", reslt_Qu2)


Q2 Result:
    book_id                title     author
0      505  Letters to the Nile  Aya Hafez
1      506  The Paper Boat Club  Aya Hafez


In [118]:
QU3_sql = """
SELECT b.title, COUNT(c.checkout_id) as checkouts
FROM checkouts c
JOIN books b ON c.book_id = b.book_id
GROUP BY b.book_id
ORDER BY checkouts DESC
LIMIT 5;
"""
reslt_Qu3 = pd.read_sql_query(QU3_sql, conn)
print("\nQ3 Result:\n", reslt_Qu3)


Q3 Result:
                     title  checkouts
0         The Silver Kite         57
1   Fossils and Fireflies         55
2  Circuits for Beginners         46
3        Kites Over Cairo         38
4    Storms and Sailboats         25


In [119]:
Qu4_sql = """
SELECT m.first_name, m.last_name, 
COUNT(c.checkout_id) as checkouts
FROM checkouts c 
JOIN members m ON c.member_id = m.member_id
GROUP BY m.member_id 
ORDER BY checkouts DESC 
LIMIT 10;
"""
result_QU4 = pd.read_sql_query(Qu4_sql, conn)
print("\nQ4 Result:\n", result_QU4)


Q4 Result:
   first_name last_name  checkouts
0        Aya     Wahba         25
1     Sherif     Saleh         21
2       Ziad     Saleh         19
3    Mostafa     Fouad         18
4       Nour     Nabil         18
5       Adam     Fahmy         17
6    Youssef    Hegazy         17
7      Ahmed    Shafik         17
8       Sara    Rashad         16
9       Reem     Osman         16


In [120]:
Q5_sql = """
SELECT c.*, m.neighborhood
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE m.neighborhood = 'Maadi'
ORDER BY c.checkout_date DESC;
"""

res_q5 = pd.read_sql_query(Q5_sql, conn)
rest_Q5 = res_q5.iloc[10:]
print("\nQ5 Result:\n", rest_Q5.head())




Q5 Result:
     checkout_id  member_id  book_id checkout_date return_date neighborhood
10         9103       1003      513    2025-09-04  2025-09-25        Maadi
11         9081       1017      502    2025-08-25  2025-09-17        Maadi
12         9001       1008      501    2025-08-23  2025-08-28        Maadi
13         9050       1003      521    2025-08-21         NaN        Maadi
14         9085       1018      525    2025-08-19  2025-09-18        Maadi


In [121]:
Db_members = pd.read_sql_query("SELECT * FROM members", conn)
Db_books = pd.read_sql_query("SELECT * FROM books", conn)
Db_checkouts = pd.read_sql_query("SELECT * FROM checkouts", conn)
Json_books = pd.read_json("download.json")
Html_kickoff = pd.read_html("download.html")[0]
Html_kickoff.columns = ["member_id", "book_id", "checkout_date"]
Html_kickoff["return_date"] = None
allbooks = pd.merge(Db_books, Json_books, how="left", on="book_id")
merged_db = Db_checkouts.merge(Db_members, on="member_id", how="left").merge(allbooks, on="book_id", how="left")
merged_Html = Html_kickoff.merge(Db_members, on="member_id", how="left").merge(allbooks, on="book_id", how="left")
final_df = pd.concat([merged_db, merged_Html], ignore_index=True)
checkout_counts = final_df["member_id"].value_counts()
final_df["member_total_checkouts"] = final_df["member_id"].map(checkout_counts)
final_df.to_csv("task1_combined_data.csv", index=False)